# NC-02: Graph Construction & MSD Feature Encoding

Rebuilds the CellComplex from OBJ, constructs a room adjacency graph,
encodes MSD-compatible node and edge features, then exports the graph
as CSV files ready for spatial analysis and node classification.

**Run after NC-01** — this notebook re-imports the same OBJ files.

In [1]:
from pathlib import Path
from collections import defaultdict

from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Graph import Graph
from topologicpy.Edge import Edge
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Vertex import Vertex
import pandas as pd

e:\softwares-4\graph-ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Paths

In [2]:
BASE          = Path(r'E:\softwares-4\graph-ml\assign-04-node-classification')
GEOMETRY_PATH = BASE / 'geometry'
GRAPHS_PATH   = BASE / 'graphs'
GRAPHS_PATH.mkdir(exist_ok=True)

VALID_ROOM_TYPES = {
    'bedroom', 'livingroom', 'kitchen', 'dining',
    'corridor', 'stairs', 'storeroom', 'bathroom', 'balcony'
}

print('Geometry :', GEOMETRY_PATH)
print('Graphs   :', GRAPHS_PATH)

Geometry : E:\softwares-4\graph-ml\assign-04-node-classification\geometry
Graphs   : E:\softwares-4\graph-ml\assign-04-node-classification\graphs


## 1. Rebuild CellComplex from OBJ

Replicates the key steps from NC-01: parse OBJ groups, split faces into
connected components (Union-Find), build a `Cell` per component, then
merge into a `CellComplex` with `room_type` dictionaries on each cell.

In [3]:
def faces_to_cells(faces, label):
    if not faces:
        return []
    n = len(faces)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    face_verts = []
    for f in faces:
        vset = set()
        for e in (Topology.Edges(f) or []):
            for v in (Topology.Vertices(e) or []):
                vset.add((round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2)))
        face_verts.append(vset)

    for i in range(n):
        for j in range(i + 1, n):
            if face_verts[i] & face_verts[j]:
                union(i, j)

    components = defaultdict(list)
    for i, f in enumerate(faces):
        components[find(i)].append(f)

    result = []
    for comp in components.values():
        cell = Cell.ByFaces(comp)
        if cell is None:
            print('    WARN Cell.ByFaces failed:', label, len(comp), 'faces')
            continue
        result.append(cell)
    return result


objects = Topology.ByOBJPath(str(GEOMETRY_PATH / 'rooms.obj'))
print('Objects returned:', len(objects))

cells     = []
selectors = []

for obj in objects:
    d         = Topology.Dictionary(obj)
    name      = Dictionary.ValueAtKey(d, 'name') or ''
    room_type = name.strip().split()[-1] if name else ''
    if room_type not in VALID_ROOM_TYPES:
        continue
    faces      = Topology.Faces(obj) or []
    room_cells = faces_to_cells(faces, room_type)
    for cell in room_cells:
        s = Topology.InternalVertex(cell)
        s = Topology.SetDictionary(s, Dictionary.ByKeysValues(['room_type'], [room_type]))
        selectors.append(s)
        cells.append(cell)
    print(' ', room_type, '->', len(room_cells), 'cell(s)')

print()
print('Total cells:', len(cells))

Objects returned: 7
  bedroom -> 3 cell(s)
  livingroom -> 1 cell(s)
  kitchen -> 1 cell(s)
  bathroom -> 2 cell(s)
  corridor -> 2 cell(s)
  dining -> 1 cell(s)

Total cells: 10


## 2. Build CellComplex

In [4]:
cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True)

cc_cells = Topology.Cells(cc) or []
print('CellComplex cells:', len(cc_cells))
print('Room types assigned:')
for cell in cc_cells:
    rt = Dictionary.ValueAtKey(Topology.Dictionary(cell), 'room_type')
    print('  ', rt)

CellComplex cells: 10
Room types assigned:
   bedroom
   bathroom
   corridor
   bedroom
   dining
   livingroom
   kitchen
   corridor
   bathroom
   bedroom


## 3. Build adjacency graph

`direct=True` creates an edge between every pair of cells that share a face,
giving us the room adjacency graph needed for node classification.

In [5]:
graph = Graph.ByTopology(
    cc,
    direct=True,
    viaSharedTopologies=False,
    viaSharedApertures=False,
    toExteriorApertures=False,
    toExteriorTopologies=False,
)

vertices = Graph.Vertices(graph) or []
edges    = Graph.Edges(graph)    or []
print('Graph vertices:', len(vertices))
print('Graph edges   :', len(edges))

# Shared-face adjacency implies a door connection; default door_type='door'
for e in edges:
    Topology.SetDictionary(e, Dictionary.ByKeysValues(['door_type'], ['door']))
print("Assigned door_type='door' to all edges")

Graph vertices: 10
Graph edges   : 13
Assigned door_type='door' to all edges


## 4. Encode MSD features

Loads `EncodeMSDGraphFeatures` and `CheckMSDGraphPreparation` from the
provided **S06-15B** notebook, then encodes node labels (0-8), zoning
one-hot features, and edge connectivity features.

In [6]:
# Source: S06-15B GML Prepare Graph for Node Classification
# Functions used verbatim from the provided course material

DOOR_TYPE_TO_CONNECTIVITY = {
    "passage": 0, "door": 1, "entrance_door": 2,
    "entrance": 2, "entrance door": 2, "entrance-door": 2,
}
CONNECTIVITY_TO_DOOR_TYPE = {0: "passage", 1: "door", 2: "entrance_door"}
ROOM_TYPE_TO_LABEL = {
    "bedroom": 0, "livingroom": 1, "living_room": 1, "living room": 1,
    "kitchen": 2, "dining": 3, "corridor": 4, "stairs": 5,
    "stair": 5, "staircase": 5, "storeroom": 6, "store_room": 6,
    "store room": 6, "storage": 6, "bathroom": 7, "balcony": 8,
}
ROOM_TYPE_TO_ZONING = {
    "bedroom": 0,
    "livingroom": 1, "living_room": 1, "living room": 1,
    "kitchen": 1, "dining": 1, "corridor": 1,
    "stairs": 2, "stair": 2, "staircase": 2,
    "storeroom": 2, "store_room": 2, "store room": 2, "storage": 2,
    "bathroom": 2, "balcony": 3,
}
EDGE_FEATURE_KEYS = ["feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"]
NODE_ZONING_FEATURE_KEYS = ["feat_zoning_type_0", "feat_zoning_type_1", "feat_zoning_type_2", "feat_zoning_type_3"]
NODE_CONNECTIVITY_FEATURE_KEYS = ["feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"]

def _clean_string(value):
    if value is None: return None
    return str(value).strip().lower().replace("-", "_")

def _one_hot(index, length):
    values = [0] * length; values[int(index)] = 1; return values

def _merge_dictionary(topology, keys, values):
    old_d = Topology.Dictionary(topology)
    new_d = Dictionary.ByKeysValues(keys, values)
    merged = Dictionary.ByMergedDictionaries([old_d, new_d]) if old_d else new_d
    Topology.SetDictionary(topology, merged)
    return topology

def _dictionary_value(topology, key, default=None):
    d = Topology.Dictionary(topology)
    return Dictionary.ValueAtKey(d, key, default)

def _vertex_xyz_key(vertex, mantissa=6):
    return (round(float(Vertex.X(vertex)), mantissa),
            round(float(Vertex.Y(vertex)), mantissa),
            round(float(Vertex.Z(vertex)), mantissa))

def _find_vertex_index(vertex, vertices, xyz_index, mantissa=6, tolerance=0.0001):
    key = _vertex_xyz_key(vertex, mantissa=mantissa)
    for i in xyz_index.get(key, []):
        try:
            if Topology.IsSame(vertex, vertices[i]): return i
        except Exception: pass
        try:
            if Vertex.Distance(vertex, vertices[i]) <= tolerance: return i
        except Exception: pass
    for i, v in enumerate(vertices):
        try:
            if Topology.IsSame(vertex, v): return i
        except Exception: pass
        try:
            if Vertex.Distance(vertex, v) <= tolerance: return i
        except Exception: pass
    return None

def _edge_connectivity_index(edge, doorTypeKey="door_type"):
    door_type = _clean_string(_dictionary_value(edge, doorTypeKey, None))
    if door_type in DOOR_TYPE_TO_CONNECTIVITY:
        return DOOR_TYPE_TO_CONNECTIVITY[door_type]
    value = _dictionary_value(edge, "connectivity", None)
    if value is not None:
        try:
            v = int(value)
            if v in (0, 1, 2): return v
        except Exception: pass
    return None


def EncodeMSDGraphFeatures(graph, roomTypeKey="room_type", doorTypeKey="door_type",
                           overwrite=True, normalizeNodeConnectivity=True,
                           mantissa=6, tolerance=0.0001, silent=False,
                           writeReadableAliases=False):
    if graph is None:
        if not silent: print("EncodeMSDGraphFeatures - Error: graph is None.")
        return None
    vertices = Graph.Vertices(graph)
    edges    = Graph.Edges(graph) or []
    if not vertices:
        if not silent: print("EncodeMSDGraphFeatures - Error: no vertices.")
        return graph
    edge_connectivity = []
    for edge in edges:
        c_index = _edge_connectivity_index(edge, doorTypeKey=doorTypeKey)
        if c_index is None:
            edge_connectivity.append(None)
            if not silent: print("EncodeMSDGraphFeatures - Warning: edge missing door_type.")
            continue
        one_hot = _one_hot(c_index, 3)
        keys   = ["connectivity"] + EDGE_FEATURE_KEYS + [doorTypeKey]
        values = [int(c_index)] + one_hot + [CONNECTIVITY_TO_DOOR_TYPE[int(c_index)]]
        _merge_dictionary(edge, keys, values)
        edge_connectivity.append(c_index)
    xyz_index = {}
    for i, vertex in enumerate(vertices):
        key = _vertex_xyz_key(vertex, mantissa=mantissa)
        xyz_index.setdefault(key, []).append(i)
    incident_counts = [[0.0, 0.0, 0.0] for _ in vertices]
    for edge, c_index in zip(edges, edge_connectivity):
        if c_index is None: continue
        try:
            sv = Edge.StartVertex(edge)
            ev = Edge.EndVertex(edge)
        except Exception: continue
        si = _find_vertex_index(sv, vertices, xyz_index, mantissa=mantissa, tolerance=tolerance)
        ei = _find_vertex_index(ev, vertices, xyz_index, mantissa=mantissa, tolerance=tolerance)
        if si is not None: incident_counts[si][c_index] += 1.0
        if ei is not None and ei != si: incident_counts[ei][c_index] += 1.0
    for i, vertex in enumerate(vertices):
        room_type_raw = _dictionary_value(vertex, roomTypeKey, None)
        room_type = _clean_string(room_type_raw)
        if room_type not in ROOM_TYPE_TO_LABEL:
            if not silent: print("EncodeMSDGraphFeatures - Warning: unsupported room_type", repr(room_type_raw))
            label = -1; zoning_type = -1; zoning_features = [0, 0, 0, 0]
        else:
            label = ROOM_TYPE_TO_LABEL[room_type]
            zoning_type = ROOM_TYPE_TO_ZONING[room_type]
            zoning_features = _one_hot(zoning_type, 4)
        conn_vals = incident_counts[i]
        if normalizeNodeConnectivity:
            total = sum(conn_vals)
            conn_vals = [round(v / total, mantissa) for v in conn_vals] if total > 0 else [0.0, 0.0, 0.0]
        else:
            conn_vals = [round(v, mantissa) for v in conn_vals]
        keys   = [roomTypeKey, "label", "zoning_type"] + NODE_ZONING_FEATURE_KEYS + NODE_CONNECTIVITY_FEATURE_KEYS
        values = [room_type if room_type else room_type_raw, int(label), int(zoning_type)] + zoning_features + conn_vals
        _merge_dictionary(vertex, keys, values)
    return graph


def CheckMSDGraphPreparation(graph, roomTypeKey="room_type", doorTypeKey="door_type", silent=False):
    vertices = Graph.Vertices(graph) if graph else []
    edges    = Graph.Edges(graph)    if graph else []
    unsupported_room_types = []
    unsupported_door_types = []
    missing_room_type_count = 0
    missing_door_type_count = 0
    for vertex in vertices:
        value = _dictionary_value(vertex, roomTypeKey, None)
        clean = _clean_string(value)
        if clean is None: missing_room_type_count += 1
        elif clean not in ROOM_TYPE_TO_LABEL: unsupported_room_types.append(value)
    for edge in (edges or []):
        value = _dictionary_value(edge, doorTypeKey, None)
        clean = _clean_string(value)
        if clean is None: missing_door_type_count += 1
        elif clean not in DOOR_TYPE_TO_CONNECTIVITY: unsupported_door_types.append(value)
    summary = {
        "vertex_count": len(vertices),
        "edge_count": len(edges or []),
        "missing_room_type_count": missing_room_type_count,
        "missing_door_type_count": missing_door_type_count,
        "unsupported_room_types": sorted(set(unsupported_room_types)),
        "unsupported_door_types": sorted(set(unsupported_door_types)),
    }
    if not silent:
        for key, value in summary.items(): print(str(key) + ": " + str(value))
    return summary

print("EncodeMSDGraphFeatures and CheckMSDGraphPreparation ready.")

EncodeMSDGraphFeatures and CheckMSDGraphPreparation ready.


In [7]:
print('--- Pre-encoding check ---')
CheckMSDGraphPreparation(graph)

print()
print('--- Encoding features ---')
graph = EncodeMSDGraphFeatures(graph, silent=False)
print('Done.')

--- Pre-encoding check ---
vertex_count: 10
edge_count: 13
missing_room_type_count: 0
missing_door_type_count: 0
unsupported_room_types: []
unsupported_door_types: []

--- Encoding features ---
Done.


## 5. Inspect encoded dictionaries

In [8]:
vertices = Graph.Vertices(graph) or []
edges    = Graph.Edges(graph)    or []

if vertices:
    print('First vertex dictionary:')
    d = Topology.Dictionary(vertices[0])
    for k in (Dictionary.Keys(d) or []):
        print(f'  {k}: {Dictionary.ValueAtKey(d, k)}')

if edges:
    print()
    print('First edge dictionary:')
    d = Topology.Dictionary(edges[0])
    for k in (Dictionary.Keys(d) or []):
        print(f'  {k}: {Dictionary.ValueAtKey(d, k)}')

First vertex dictionary:
  aabb: [5511.986328, 2240.0, 0.0, 11511.986328, 5240.0, 3000.0]
  category: 0
  feat_connectivity_0: 0.0
  feat_connectivity_1: 1.0
  feat_connectivity_2: 0.0
  feat_zoning_type_0: 1
  feat_zoning_type_1: 0
  feat_zoning_type_2: 0
  feat_zoning_type_3: 0
  index: 0
  label: 0
  room_type: bedroom
  zoning_type: 0

First edge dictionary:
  connectivity: 1
  door_type: door
  feat_connectivity_0: 0
  feat_connectivity_1: 1
  feat_connectivity_2: 0


## 6. Export to CSV

Exports `graphs.csv`, `nodes.csv`, and `edges.csv` to `graphs/`,
then stamps all nodes as test-only (inference — no training split).

In [9]:
NODE_FEATURE_KEYS = [
    'feat_zoning_type_0', 'feat_zoning_type_1',
    'feat_zoning_type_2', 'feat_zoning_type_3',
    'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2',
]

status = Graph.ExportToCSV(
    graph,
    path=str(GRAPHS_PATH),
    nodeLabelKey='label',
    nodeFeaturesKeys=NODE_FEATURE_KEYS,
    overwrite=True,
)
print('ExportToCSV status:', status)

# All nodes are test nodes — inference only, no training split
nodes_csv = GRAPHS_PATH / 'nodes.csv'
nodes_df  = pd.read_csv(nodes_csv)
nodes_df['train_mask'] = False
nodes_df['val_mask']   = False
nodes_df['test_mask']  = True
nodes_df.to_csv(nodes_csv, index=False)
print('Masks written to nodes.csv')

ExportToCSV status: True
Masks written to nodes.csv


## 7. Verify output

In [10]:
for fname in ('graphs.csv', 'nodes.csv', 'edges.csv'):
    p = GRAPHS_PATH / fname
    if p.exists():
        df = pd.read_csv(p)
        print(f'{fname}: {df.shape[0]} rows x {df.shape[1]} cols')
        print('  columns:', list(df.columns))
        print()
    else:
        print(f'MISSING: {fname}')

graphs.csv: 1 rows x 2 cols
  columns: ['graph_id', 'label']

nodes.csv: 10 rows x 16 cols
  columns: ['graph_id', 'node_id', 'label', 'train_mask', 'val_mask', 'test_mask', 'feat_feat_zoning_type_0', 'feat_feat_zoning_type_1', 'feat_feat_zoning_type_2', 'feat_feat_zoning_type_3', 'feat_feat_connectivity_0', 'feat_feat_connectivity_1', 'feat_feat_connectivity_2', 'X', 'Y', 'Z']

edges.csv: 26 rows x 7 cols
  columns: ['graph_id', 'src_id', 'dst_id', 'label', 'train_mask', 'val_mask', 'test_mask']

